# YouTube Recommendation System

**Pipeline:**
1. Collect data (YouTube Data API v3)
2. Preprocess & clean (glob multiple CSV shards)
3. Parse watch history (Google Takeout)
4. **Enrich catalog** with watched videos missing from catalog
5. Build embeddings (BGE-M3) + FAISS index
6. Baseline: cosine-similarity recommendation
7. Main model: CatBoost two-stage ranking
8. **Evaluation**: Hit Rate / Precision / Recall / NDCG @ K

## Step 1 — Data Collection

Получает видео через YouTube Data API v3.  
API-ключ: [Google Cloud Console](https://console.cloud.google.com) → APIs & Services → YouTube Data API v3 → Credentials.

In [ ]:
import time
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

API_KEY = "YOUR_YOUTUBE_API_KEY"
TARGET_VIDEOS = 65000
SAVE_EVERY = 500

youtube = build("youtube", "v3", developerKey=API_KEY)

SEARCH_QUERIES = [

    # ---------------- ML ----------------
    # "машинное обучение",
    # "machine learning tutorial",
    # "deep learning tutorial",
    # "нейронные сети",
    # "data science tutorial",
    # "python machine learning",
    # "ml system design",
    # "ml interview",
    # "xgboost tutorial",
    # "catboost tutorial",
    # "pytorch tutorial",
    # "tensorflow tutorial",
    # "ml inside",
    # "ml inside machine learning",
    # "ml inside deep learning",
    # "ml engineering",
    # "llm tutorial",
    # "nlp tutorial",
    # "computer vision tutorial",
    # "reinforcement learning tutorial",

    # ---------------- Programming ----------------
    # "python programming",
    # "java tutorial",
    # "golang tutorial",
    # "rust tutorial",
    # "c++ tutorial",
    # "javascript tutorial",
    # "typescript tutorial",
    # "backend development",
    # "system design",
    # "leetcode solutions",
    # "algorithms and data structures",
    # "django tutorial",
    # "fastapi tutorial",
    # "spring boot tutorial",
    # "react tutorial",
    # "node js tutorial",
    # "sql tutorial",
    # "как писать на perl",

    # ---------------- Entertainment ----------------
    # "смешные видео",
    # "юмор",
    # "memes compilation",
    # "funny moments",
    # "развлекательный контент",
    # "импровизация",
    # "стендап",

    # ---------------- Dota2 ----------------
    # "dota 2 highlights",
    # "dota 2 guide",
    # "dota 2 funny moments",
    # "дота 2 стрим", 
    # "дота 2 патч обзор",

    # ---------------- CS2 ----------------
    # "cs2 highlights",
    # "cs2 funny moments",
    # "cs2 gameplay",
    # "counter strike 2 guide",
    # "кс 2 моменты",

    # ---------------- Isaac ----------------
    # "binding of isaac gameplay",
    # "binding of isaac guide",
    # "binding of isaac funny moments",

    # ---------------- Other Games ----------------
    # "elden ring gameplay",
    # "minecraft gameplay",
    # "terraria gameplay",
    # "hades gameplay",
    # "baldurs gate 3 gameplay",
    # "witcher gameplay",
    # "играю в разные игры", 65
    # "прохождение игры",
    # "minecraft миниигры",

    # ---------------- Biology ----------------
    # "биология животных",
    # "редкие 

    # ---------------- Sports ----------------
    # "football highlights",животные",
    # "интересные животные",
    # "документальный фильм животные",
    # "biology documentary animals",
    # "ты леминг и это вся твоя жизнь",

    # ---------------- Red pandas ----------------
    # "red panda",
    # "красная панда",
    # "funny red panda",
    # "mma highlights",
    # "ufc highlights",
    # "basketball highlights",
    # "workout training",
    # "техника выполнения упраженний",
    # "программы тренировок в зале",

    # ---------------- Weight loss ----------------
    # "как похудеть",
    # "weight loss tips",
    # "fat loss",
    # "nutrition for weight loss",

    # ---------------- Health ----------------
    # "здоровье",
    # "healthy lifestyle",
    # "how to be healthy",
    # "doctor explains health",
    # "похудение",

    # ---------------- Sleep ----------------
    # "как улучшить сон",
    # "sleep science",
    # "better sleep tips",
    # "sleep optimization",
    # "подкаст о сне",

    # ---------------- News ----------------
    # "новости",
    # "world news",
    # "экономические новости",
    # "политические новости",
    # "новостной блог",

    # ---------------- Economics ----------------
    # "economics explained",
    # "макроэкономика",
    # "финансы",
    # "investment basics",
    # "простоая экономика",
    # "что с экономикой россии",
    # "во что инвестировать",
    # "разбор финансовых инструментов",

    # ---------------- Educational entertainment ----------------
    # "фаиб",
    # "geo youtube channel",
    # "mygap",
    # "егор максимов",
    # "история и политика объяснение",
    # "interesting science explained",
    # "физические эксперементы",
    # "химические эксперементы",
    # "програмирования роботов",
    # "создание умных устройств на ардуино",

    # ---------------- Movies ----------------
    # "movie review",
    # "film analysis",
    # "лучшие фильмы",
    # "разбор фильмов",
    # "топ 10 фильмов",
    # "как снять свой фильм",
    # "топ лучших камер для новичка",
    # "основы композиции",
    # "основы света",
    # "как псать сценарий",
    # "где брать референсы для съёмки",
    # "обучение видеосъёмке",
    # "как выбрать камеру",

    # ---------------- Series ----------------
    # "tv series review",
    # "лучшие сериалы",
    # "series analysis",
    # "топ 10 сериалов",

    # ---------------- Series ----------------
    # "рецепты",
    # "меню на неделю",
    # "рецепты из курицы",
    # "ужин за 500 рублей",
    # "живу менеделю на 1000 рублей",
    # "заготовки на месяц",
    # "обзор ресторана",
    # "высокая кухня",
    # "обзор доставки еды",
    # "как производят еду",
]


def search_videos(query, page_token=None):
    return youtube.search().list(
        q=query, part="snippet", type="video",
        maxResults=50, pageToken=page_token, relevanceLanguage="ru",
    ).execute()


def get_video_details(video_ids):
    response = youtube.videos().list(
        part="snippet,statistics,contentDetails",
        id=",".join(video_ids),
    ).execute()
    videos = []
    for item in response["items"]:
        snippet = item.get("snippet", {})
        stats   = item.get("statistics", {})
        content = item.get("contentDetails", {})
        videos.append({
            "video_id":      item["id"],
            "title":         snippet.get("title"),
            "description":   snippet.get("description"),
            "channel_title": snippet.get("channelTitle"),
            "published_at":  snippet.get("publishedAt"),
            "tags":          snippet.get("tags", []),
            "category_id":   snippet.get("categoryId"),
            "duration":      content.get("duration"),
            "view_count":    stats.get("viewCount"),
            "like_count":    stats.get("likeCount"),
            "comment_count": stats.get("commentCount"),
        })
    return videos


def save_progress(all_videos, filename="youtube_large_dataset.csv"):
    pd.DataFrame(list(all_videos.values())).to_csv(filename, index=False)
    print(f"Saved {len(all_videos)} → {filename}")


def collect_dataset():
    all_videos: dict = {}
    current_idx = 0
    try:
        for query_idx, query in enumerate(SEARCH_QUERIES):
            current_idx = query_idx
            print(f"[{query_idx}/{len(SEARCH_QUERIES)}] {query}")
            page_token = None
            while True:
                response  = search_videos(query, page_token=page_token)
                video_ids = [item["id"]["videoId"] for item in response["items"]]
                for video in get_video_details(video_ids):
                    if video["video_id"] not in all_videos:
                        all_videos[video["video_id"]] = video
                print(f"  Collected: {len(all_videos)}")
                if len(all_videos) % SAVE_EVERY == 0 and len(all_videos) > 0:
                    save_progress(all_videos)
                page_token = response.get("nextPageToken")
                if not page_token or len(all_videos) >= TARGET_VIDEOS:
                    break
                time.sleep(1)
            if len(all_videos) >= TARGET_VIDEOS:
                break
    except HttpError as e:
        print(f"\nAPI quota at query {current_idx}: {SEARCH_QUERIES[current_idx]}\n{e}")
    except Exception as e:
        print(f"\nError: {e}")
    finally:
        save_progress(all_videos)


collect_dataset()


## Step 2 — Preprocessing

Автоматически подхватывает все CSV-шарды (`youtube_large_dataset*.csv`), мёржит их, чистит и сохраняет в единый parquet.

In [1]:
!pip install isodate

In [ ]:
import glob
import pandas as pd
import numpy as np
import re
import ast
import isodate

csv_files = sorted(glob.glob("youtube_large_dataset*.csv"))
print(f"Found {len(csv_files)} CSV files: {csv_files}")

df = pd.concat(
    [pd.read_csv(f) for f in csv_files],
    ignore_index=True,
)
print(f"After merge: {df.shape}")

df = df.drop_duplicates(subset="video_id")
df = df.dropna(subset=["title"])
df["description"] = df["description"].fillna("")
df["tags"]        = df["tags"].fillna("[]")

for col in ["view_count", "like_count", "comment_count"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["view_count"])

def parse_duration(s):
    try:
        return isodate.parse_duration(s).total_seconds()
    except Exception:
        return np.nan

df["duration_seconds"] = df["duration"].apply(parse_duration)
df = df.dropna(subset=["duration_seconds"])
df = df[(df["duration_seconds"] >= 120) & (df["duration_seconds"] <= 18_000)]

df = df[~df["title"].str.lower().str.contains("shorts", na=False)]
bad_kw = ["розыгрыш", "казино", "ставки", "заработок без усилий", "crypto scam"]
df = df[~df["title"].str.lower().str.contains("|".join(bad_kw), na=False)]

def parse_tags(s):
    try:
        lst = ast.literal_eval(s)
        return " ".join(lst) if isinstance(lst, list) else ""
    except Exception:
        return ""

df["tags_clean"] = df["tags"].apply(parse_tags)

df["full_text"] = (
    df["title"].fillna("") + " " +
    df["description"].fillna("") + " " +
    df["tags_clean"].fillna("")
)

def clean_text(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    return re.sub(r"\s+", " ", text).strip()

df["full_text"] = df["full_text"].apply(clean_text)
df = df[df["full_text"].str.len() > 30]
df = df.reset_index(drop=True)

df.to_parquet("youtube_clean.parquet", index=False)
print(f"Final catalog: {df.shape}")
print(df[["title", "view_count", "duration_seconds"]].head(3))


Final catalog: (35695, 14)
                                               title  view_count  \
0            МАШИННОЕ ОБУЧЕНИЕ - ВСЕ ЧТО НУЖНО ЗНАТЬ    161412.0   
1  Машинное обучение. Вводная лекция. К.В. Воронц...    199970.0   
2                     Машинное обучение для чайников     77294.0   

   duration_seconds  
0            1661.0  
1            5678.0  
2             805.0  


## Step 3 — Parse Watch History

Скачай историю: [myaccount.google.com](https://myaccount.google.com) → Data & privacy → Download your data → YouTube  
→ `Takeout/YouTube/history/watch-history.html`

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import re

HISTORY_FILE = "история-просмотров.html"

with open(HISTORY_FILE, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

records = []
for item in soup.find_all(
    "div",
    {"class": "content-cell mdl-cell mdl-cell--6-col mdl-typography--body-1"},
):
    links = item.find_all("a")
    if not links:
        continue
    url   = links[0].get("href", "")
    match = re.search(r"v=([a-zA-Z0-9_-]+)", url)
    records.append({
        "video_title":  links[0].text,
        "video_id":     match.group(1) if match else None,
        "channel_name": links[1].text if len(links) > 1 else None,
    })

history_df = (
    pd.DataFrame(records)
    .dropna(subset=["video_id"])
    .drop_duplicates(subset=["video_id"])
    .reset_index(drop=True)
)

history_df.to_parquet("parsed_watch_history.parquet", index=False)
print(f"Parsed {len(history_df)} unique watched videos")
print(history_df.head(3))


## Step 4 — Enrich Catalog with Watched Videos

Добирает через API все просмотренные тобой видео, которых нет в собранном каталоге. Это решает проблему маленького пересечения (было 241 → станет ~2500).

In [62]:
import pandas as pd

MIN_DURATION_SECONDS = 180 


watch_df = pd.read_parquet("parsed_watch_history.parquet")
videos_df = pd.read_parquet("youtube_clean.parquet")

print("Before filtering:")
print(f"watch_df:  {len(watch_df):,}")
print(f"videos_df: {len(videos_df):,}")


videos_filtered = videos_df[
    videos_df["duration_seconds"] >= MIN_DURATION_SECONDS
].copy()

print("\nAfter video filtering:")
print(f"videos_filtered: {len(videos_filtered):,}")


# Предполагается, что ключ — video_id
valid_video_ids = set(videos_filtered["video_id"])

watch_filtered = watch_df[
    watch_df["video_id"].isin(valid_video_ids)
].copy()

print(f"watch_filtered: {len(watch_filtered):,}")


removed_videos = len(videos_df) - len(videos_filtered)
removed_watch = len(watch_df) - len(watch_filtered)

print("\nRemoved:")
print(f"Videos removed: {removed_videos:,}")
print(f"Watch events removed: {removed_watch:,}")

videos_filtered.to_parquet(
    "youtube_clean_filtered.parquet",
    index=False
)

watch_filtered.to_parquet(
    "parsed_watch_history_filtered.parquet",
    index=False
)

print("\nSaved:")
print("youtube_clean_filtered.parquet")
print("parsed_watch_history_filtered.parquet")

Before filtering:
watch_df:  55,729
videos_df: 38,067

After video filtering:
videos_filtered: 33,424
watch_filtered: 408

Removed:
Videos removed: 4,643
Watch events removed: 55,321

Saved:
youtube_clean_filtered.parquet
parsed_watch_history_filtered.parquet


In [3]:
import time
import re
import ast
import isodate
import numpy as np
import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

API_KEY   = "YOUR_YOUTUBE_API_KEY"
ENRICH_N  = 408   # сколько просмотренных видео добавить в каталог

history_df = pd.read_parquet("parsed_watch_history_filtered.parquet")
videos_df  = pd.read_parquet("youtube_clean_filtered.parquet")

catalog_ids = set(videos_df["video_id"])
history_ids = list(history_df["video_id"])

missing_ids = [vid for vid in history_ids if vid and vid not in catalog_ids]
print(f"Watched videos missing from catalog: {len(missing_ids)}")

missing_ids = missing_ids[:ENRICH_N]
print(f"Will fetch: {len(missing_ids)}")

if not missing_ids:
    print("Nothing to fetch — catalog already contains all watched videos.")
else:
    youtube = build("youtube", "v3", developerKey=API_KEY)

    def fetch_batch(ids_batch):
        try:
            response = youtube.videos().list(
                part="snippet,statistics,contentDetails",
                id=",".join(ids_batch),
            ).execute()
        except HttpError as e:
            print(f"  API error: {e}")
            return []
        rows = []
        for item in response.get("items", []):
            snippet = item.get("snippet", {})
            stats   = item.get("statistics", {})
            content = item.get("contentDetails", {})
            rows.append({
                "video_id":      item["id"],
                "title":         snippet.get("title"),
                "description":   snippet.get("description"),
                "channel_title": snippet.get("channelTitle"),
                "published_at":  snippet.get("publishedAt"),
                "tags":          snippet.get("tags", []),
                "category_id":   snippet.get("categoryId"),
                "duration":      content.get("duration"),
                "view_count":    stats.get("viewCount"),
                "like_count":    stats.get("likeCount"),
                "comment_count": stats.get("commentCount"),
            })
        return rows

    all_fetched = []
    batch_size  = 50
    for i in range(0, len(missing_ids), batch_size):
        batch = missing_ids[i : i + batch_size]
        rows  = fetch_batch(batch)
        all_fetched.extend(rows)
        print(f"  Fetched {len(all_fetched)}/{len(missing_ids)}", end="\r")
        time.sleep(0.5)

    print(f"\nFetched {len(all_fetched)} videos from API")

    if all_fetched:
        new_df = pd.DataFrame(all_fetched)

        def parse_duration(s):
            try:
                return isodate.parse_duration(s).total_seconds()
            except Exception:
                return np.nan

        def parse_tags(s):
            if isinstance(s, list):
                return " ".join(s)
            try:
                lst = ast.literal_eval(str(s))
                return " ".join(lst) if isinstance(lst, list) else ""
            except Exception:
                return ""

        def clean_text(text):
            text = re.sub(r"http\S+|www\S+", "", str(text))
            return re.sub(r"\s+", " ", text).strip()

        new_df = new_df.dropna(subset=["title"])
        new_df["description"] = new_df["description"].fillna("")
        new_df["tags"]        = new_df["tags"].fillna("[]")

        for col in ["view_count", "like_count", "comment_count"]:
            new_df[col] = pd.to_numeric(new_df[col], errors="coerce").fillna(0)

        new_df["duration_seconds"] = new_df["duration"].apply(parse_duration)
        new_df = new_df.dropna(subset=["duration_seconds"])

        new_df["tags_clean"] = new_df["tags"].apply(parse_tags)
        new_df["full_text"]  = (
            new_df["title"].fillna("") + " " +
            new_df["description"].fillna("") + " " +
            new_df["tags_clean"].fillna("")
        ).apply(clean_text)

        new_df = new_df[new_df["full_text"].str.len() > 10]

        # Унифицируем типы колонок перед concat, чтобы pyarrow не падал
        STR_COLS = ["tags", "category_id", "duration"]
        for col in STR_COLS:
            if col in new_df.columns:
                new_df[col] = new_df[col].apply(
                    lambda x: str(x) if not isinstance(x, str) else x
                )
            if col in videos_df.columns:
                videos_df[col] = videos_df[col].astype(str)

        combined = (
            pd.concat([videos_df, new_df], ignore_index=True)
            .drop_duplicates(subset="video_id")
            .reset_index(drop=True)
        )

        combined.to_parquet("youtube_clean.parquet", index=False)

        print(f"Catalog before: {len(videos_df)}")
        print(f"Catalog after:  {len(combined)}")
        print(f"Added:          {len(combined) - len(videos_df)} videos")
    else:
        print("No videos fetched (possibly API quota exceeded).")


Watched videos missing from catalog: 0
Will fetch: 0
Nothing to fetch — catalog already contains all watched videos.


## Step 5 — Embeddings & FAISS Index

`BAAI/bge-m3` — мощная мультиязычная модель. `IndexFlatIP` = cosine similarity для нормализованных векторов.  
⚠️ Если менял каталог (Step 4), обязательно пересобери индекс здесь.

In [6]:
!pip install faiss-cpu sentence_transformers

In [7]:
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer

df = pd.read_parquet("youtube_clean_filtered.parquet")
print(f"Catalog size: {len(df)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

model = SentenceTransformer("BAAI/bge-m3", device=device)

embeddings = model.encode(
    df["full_text"].tolist(),
    batch_size=40,
    show_progress_bar=True,
    normalize_embeddings=True,
)
embeddings = np.array(embeddings).astype("float32")
print(f"Embeddings shape: {embeddings.shape}")

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"FAISS index: {index.ntotal} vectors")

faiss.write_index(index, "youtube_faiss.index")
print("Saved → youtube_faiss.index")


Catalog size: 33424
Device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/836 [00:00<?, ?it/s]

Embeddings shape: (33424, 1024)
FAISS index: 33424 vectors
Saved → youtube_faiss.index


## Step 6 — Baseline Recommendation *(cosine similarity)*

Средний эмбеддинг просмотренных → ближайшие в FAISS. Быстрая проверка что пайплайн работает.

In [8]:
import pandas as pd
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer

history_df = pd.read_parquet("parsed_watch_history_filtered.parquet")
videos_df  = pd.read_parquet("youtube_clean_filtered.parquet")

print(f"History: {len(history_df)}  |  Catalog: {len(videos_df)}")

watched_df = history_df.merge(videos_df, on="video_id", how="inner")
print(f"Matched: {len(watched_df)} watched videos in catalog")

if len(watched_df) == 0:
    raise ValueError("No matches — run Step 4 (enrich catalog) first.")

watched_ids = set(watched_df["video_id"])

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = SentenceTransformer("BAAI/bge-m3", device=device)

watched_emb = model.encode(
    watched_df["full_text"].tolist(),
    batch_size=40,
    normalize_embeddings=True,
    show_progress_bar=True,
)
user_embedding = np.array(watched_emb).astype("float32").mean(axis=0, keepdims=True)

index = faiss.read_index("youtube_faiss.index")
scores, indices = index.search(user_embedding, 100)

recs = []
for idx, score in zip(indices[0], scores[0]):
    row = videos_df.iloc[idx]
    if row["video_id"] in watched_ids:
        continue
    recs.append({
        "title":            row["title"],
        "channel":          row["channel_title"],
        "views":            row["view_count"],
        "similarity_score": float(score),
    })

print("\n=== BASELINE TOP-20 (cosine similarity) ===")
print(pd.DataFrame(recs).head(20).to_string(index=False))


History: 408  |  Catalog: 33424
Matched: 408 watched videos in catalog


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]


=== BASELINE TOP-20 (cosine similarity) ===
                                                                         title               channel     views  similarity_score
                       Юмор 2025. Геннадий Ветров. Сборник лучших выступлений. АРМЯНСКОЕ РАДИО СМЕХА  393915.0          0.466626
          March of Industry "Тренировочные шорты Путина" с Сибирским Леммингом       SiberianLemming    6526.0          0.466385
        ДР канала, Лемминг отвечает на любые вопросы с вебкой, Игра в подарок!       SiberianLemming    4634.0          0.463045
                    Академия Cube World: Прокачка оружия и адаптация предметов       SiberianLemming   33043.0          0.459522
(Clean) Try Not to LAUGH 😂 Challenge IMPOSSIBLE | Funny Memes Compilation 2023               TreyJam  850620.0          0.458979
                                "RimWorld стал суровее". с Сибирским Леммингом       SiberianLemming   10462.0          0.458563
                           Heroes of the Storm "Зато

## Step 7 — CatBoost Ranking *(main model)*

Двухэтапная рекомендация:
- **Stage 1** (recall): FAISS достаёт 200 кандидатов по семантике
- **Stage 2** (ranking): CatBoost ранжирует по `similarity + log_views + log_likes + log_comments + duration`

Обучение: позитивы = просмотренные видео, негативы = случайная выборка из каталога.

In [9]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00:00:0100:01


In [16]:
import os
import pandas as pd
import numpy as np
import faiss

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

history_df = pd.read_parquet("parsed_watch_history_filtered.parquet")
videos_df = pd.read_parquet("youtube_clean_filtered.parquet")

videos_df = videos_df.drop_duplicates("video_id").reset_index(drop=True).copy()
history_df = history_df.copy()

print(f"History: {len(history_df)}  |  Catalog: {len(videos_df)}")

if "video_id" not in history_df.columns or "video_id" not in videos_df.columns:
    raise ValueError("Both parquet files must contain column 'video_id'.")

time_cols = ["watch_time", "watched_at", "timestamp", "time", "date", "datetime", "activity_time"]
time_col = next((c for c in time_cols if c in history_df.columns), None)

if time_col is not None:
    history_df[time_col] = pd.to_datetime(history_df[time_col], errors="coerce")
    history_df = history_df.sort_values(time_col).reset_index(drop=True)
    print(f"Split by time column: {time_col}")
else:
    history_df = history_df.reset_index(drop=True)
    print("No time column found. Split by row order.")

watched_df = history_df.merge(videos_df[["video_id"]], on="video_id", how="inner")
print(f"Matched watched videos in catalog: {len(watched_df)}")

if len(watched_df) == 0:
    raise ValueError("No matches between history and catalog.")

split_idx = int(len(watched_df) * 0.8)
train_watch = watched_df.iloc[:split_idx].copy()
test_watch  = watched_df.iloc[split_idx:].copy()

train_ids = set(train_watch["video_id"].dropna().unique())
test_ids  = set(test_watch["video_id"].dropna().unique())
seen_ids  = train_ids | test_ids

print(f"Train watched unique: {len(train_ids)}  |  Test watched unique: {len(test_ids)}")

def load_embeddings(vdf: pd.DataFrame) -> np.ndarray:
    if "embedding" in vdf.columns:
        emb = vdf["embedding"].apply(lambda x: np.asarray(x, dtype=np.float32))
        return np.vstack(emb.values).astype(np.float32)

    index_path = "youtube_faiss.index"
    if os.path.exists(index_path):
        index = faiss.read_index(index_path)
        if index.ntotal != len(vdf):
            raise ValueError(
                f"FAISS index size ({index.ntotal}) does not match catalog size ({len(vdf)}). "
                f"Rebuild the index for the current filtered catalog."
            )
        return index.reconstruct_n(0, index.ntotal).astype(np.float32)

    raise ValueError(
        "No 'embedding' column in videos_df and 'youtube_faiss.index' was not found."
    )

all_embeddings = load_embeddings(videos_df)
print(f"Embeddings shape: {all_embeddings.shape}")

video_id_to_pos = {vid: i for i, vid in enumerate(videos_df["video_id"])}

def ids_to_positions(ids, mapping):
    return [mapping[v] for v in ids if v in mapping]

train_pos_ids = [v for v in train_ids if v in video_id_to_pos]
if len(train_pos_ids) == 0:
    raise ValueError("No train positives found in catalog after filtering.")

train_pos_pos = ids_to_positions(train_pos_ids, video_id_to_pos)
positive_df = videos_df.iloc[train_pos_pos].copy().reset_index(drop=True)
positive_df["label"] = 1

pos_embeddings = all_embeddings[train_pos_pos]
user_embedding = pos_embeddings.mean(axis=0, keepdims=True).astype(np.float32)

faiss.normalize_L2(user_embedding)

candidate_df = videos_df[~videos_df["video_id"].isin(seen_ids)].copy().reset_index(drop=True)
candidate_pos = ids_to_positions(candidate_df["video_id"].tolist(), video_id_to_pos)

if len(candidate_df) == 0:
    raise ValueError("Candidate pool is empty after excluding watched videos.")

candidate_embeddings = all_embeddings[candidate_pos].astype(np.float32).copy()
faiss.normalize_L2(candidate_embeddings)

print(f"Candidate pool: {len(candidate_df)}")

dim = candidate_embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(candidate_embeddings)

def make_features(df: pd.DataFrame, sims: np.ndarray) -> pd.DataFrame:
    d = df.copy().reset_index(drop=True)
    d["similarity_score"] = sims.astype(np.float32)

    for col in ["view_count", "like_count", "comment_count", "duration_seconds"]:
        if col not in d.columns:
            d[col] = 0
        d[col] = pd.to_numeric(d[col], errors="coerce").fillna(0)

    d["log_views"] = np.log1p(d["view_count"])
    d["log_likes"] = np.log1p(d["like_count"])
    d["log_comments"] = np.log1p(d["comment_count"])
    return d

pos_sims = cosine_similarity(pos_embeddings, user_embedding).flatten()
pos_feat = make_features(positive_df, pos_sims)

n_neg = min(len(candidate_df), max(len(pos_feat) * 5, 1))
negative_df = candidate_df.sample(n=n_neg, random_state=42).copy().reset_index(drop=True)
negative_df["label"] = 0

neg_pos = ids_to_positions(negative_df["video_id"].tolist(), video_id_to_pos)
neg_embeddings = all_embeddings[neg_pos]
neg_sims = cosine_similarity(neg_embeddings, user_embedding).flatten()
neg_feat = make_features(negative_df, neg_sims)

train_data = pd.concat([pos_feat, neg_feat], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

FEATURES = ["similarity_score", "log_views", "log_likes", "log_comments", "duration_seconds"]
X = train_data[FEATURES]
y = train_data["label"]

print(f"Training rows: {len(train_data)}  (pos={int(y.sum())}, neg={int((1 - y).sum())})")

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cb_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=50
)

cb_model.fit(X_train, y_train, eval_set=(X_valid, y_valid), use_best_model=True)

auc = roc_auc_score(y_valid, cb_model.predict_proba(X_valid)[:, 1])
print(f"\nROC-AUC on validation: {auc:.4f}")

TOP_RETRIEVAL = min(200, len(candidate_df))
cand_scores, cand_indices = index.search(user_embedding, TOP_RETRIEVAL)

candidate_recs = candidate_df.iloc[cand_indices[0]].copy().reset_index(drop=True)
candidate_recs_pos = ids_to_positions(candidate_recs["video_id"].tolist(), video_id_to_pos)
candidate_recs_embeddings = all_embeddings[candidate_recs_pos]

candidate_sims = cosine_similarity(candidate_recs_embeddings, user_embedding).flatten()
cand_feat = make_features(candidate_recs, candidate_sims)
cand_feat["rank_score"] = cb_model.predict_proba(cand_feat[FEATURES])[:, 1]

for col in ["title", "channel_title", "view_count"]:
    if col not in cand_feat.columns:
        cand_feat[col] = ""

final_recs = cand_feat.sort_values("rank_score", ascending=False).head(20)

cb_model.save_model("catboost_ranker_leakfree.cbm")

print("\n=== TOP-20 RECOMMENDATIONS (Leak-free CatBoost ranking) ===")
print(final_recs[["title", "channel_title", "rank_score", "view_count"]].to_string(index=False))

History: 408  |  Catalog: 33424
No time column found. Split by row order.
Matched watched videos in catalog: 408
Train watched unique: 326  |  Test watched unique: 82
Embeddings shape: (33424, 1024)
Candidate pool: 33016
Training rows: 1956  (pos=326, neg=1630)
0:	test: 0.7846154	best: 0.7846154 (0)	total: 3.46ms	remaining: 1.03s
50:	test: 0.8456363	best: 0.8456363 (50)	total: 120ms	remaining: 588ms
100:	test: 0.8549047	best: 0.8554693 (99)	total: 210ms	remaining: 414ms
150:	test: 0.8569278	best: 0.8575865 (145)	total: 296ms	remaining: 292ms
200:	test: 0.8582922	best: 0.8591861 (173)	total: 383ms	remaining: 189ms
250:	test: 0.8583863	best: 0.8600329 (215)	total: 470ms	remaining: 91.7ms
299:	test: 0.8578687	best: 0.8600329 (215)	total: 554ms	remaining: 0us

bestTest = 0.8600329334
bestIteration = 215

Shrink model to first 216 iterations.

ROC-AUC on validation: 0.8600

=== TOP-20 RECOMMENDATIONS (Leak-free CatBoost ranking) ===
                                                          

## Step 8 — Evaluation

**Leave-20%-out**: 80% просмотров → профиль пользователя, 20% → тестовый набор.  
Проверяем, попадают ли тестовые видео в топ-K рекомендаций.

| Метрика | Что измеряет |
|---|---|
| **Hit Rate@K** | Хотя бы одно тестовое видео в топ-K |
| **Precision@K** | Доля релевантных среди топ-K |
| **Recall@K** | Доля тестовых видео, попавших в топ-K |
| **NDCG@K** | То же что Recall, но штрафует за низкую позицию |

In [17]:
"""
Оценка рекомендательных моделей.

Схема: leave-20%-out
  - Разбиваем просмотренные видео на train (80%) и test (20%)
  - Строим профиль пользователя ТОЛЬКО на train
  - Получаем рекомендации (не считая train)
  - Проверяем, попали ли test-видео в топ-K

Метрики:
  Hit Rate@K   — хотя бы одно test-видео в топ-K
  Precision@K  — доля релевантных среди топ-K
  Recall@K     — доля test-видео, попавших в топ-K
  NDCG@K       — учитывает позицию (чем выше — тем лучше)
"""
import pandas as pd
import numpy as np
import faiss
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier

# ─────────────────────────────────────────────────────────────────────────────
# LOAD
# ─────────────────────────────────────────────────────────────────────────────
history_df = pd.read_parquet("parsed_watch_history_filtered.parquet")
videos_df  = pd.read_parquet("youtube_clean_filtered.parquet")

watched_df = history_df.merge(videos_df, on="video_id", how="inner")
print(f"Matched watched videos: {len(watched_df)}")

if len(watched_df) < 10:
    raise ValueError("Слишком мало совпадений для оценки. Запусти Step 4 (enrich catalog).")

index = faiss.read_index("youtube_faiss.index")
all_embeddings = np.array(index.reconstruct_n(0, index.ntotal)).astype("float32")

video_id_to_idx = {vid: i for i, vid in enumerate(videos_df["video_id"])}

train_watched, test_watched = train_test_split(
    watched_df, test_size=0.2, random_state=42
)

train_ids = set(train_watched["video_id"])
test_ids  = set(test_watched["video_id"])

print(f"Train watched: {len(train_ids)}  |  Test watched: {len(test_ids)}")

train_positions = [video_id_to_idx[v] for v in train_watched["video_id"] if v in video_id_to_idx]
user_embedding  = all_embeddings[train_positions].mean(axis=0, keepdims=True)

def hit_rate_at_k(recs, relevant, k):
    return int(bool(set(recs[:k]) & relevant))

def precision_at_k(recs, relevant, k):
    return len(set(recs[:k]) & relevant) / k

def recall_at_k(recs, relevant, k):
    if not relevant:
        return 0.0
    return len(set(recs[:k]) & relevant) / len(relevant)

def ndcg_at_k(recs, relevant, k):
    dcg  = sum(1.0 / np.log2(i + 2) for i, v in enumerate(recs[:k]) if v in relevant)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate(rec_ids, test_ids, ks=(10, 20, 50)):
    print(f"\n{'K':>4}  {'Hit Rate':>9}  {'Precision':>10}  {'Recall':>8}  {'NDCG':>8}")
    print("-" * 46)
    for k in ks:
        hr = hit_rate_at_k(rec_ids, test_ids, k)
        p  = precision_at_k(rec_ids, test_ids, k)
        r  = recall_at_k(rec_ids, test_ids, k)
        n  = ndcg_at_k(rec_ids, test_ids, k)
        print(f"{k:>4}  {hr:>9.3f}  {p:>10.4f}  {r:>8.4f}  {n:>8.4f}")

RETRIEVE_K = 500   # сколько кандидатов берём из FAISS

scores_arr, indices_arr = index.search(user_embedding, RETRIEVE_K + len(train_ids))

baseline_recs = []
for idx in indices_arr[0]:
    vid = videos_df.iloc[idx]["video_id"]
    if vid in train_ids:
        continue
    baseline_recs.append(vid)
    if len(baseline_recs) >= RETRIEVE_K:
        break

print("\n══════════════════════════════════════")
print("  BASELINE (cosine similarity)")
print("══════════════════════════════════════")
evaluate(baseline_recs, test_ids)

FEATURES = ["similarity_score", "log_views", "log_likes", "log_comments", "duration_seconds"]

def make_features(df, sims):
    d = df.copy().reset_index(drop=True)
    d["similarity_score"] = sims
    for col in ["view_count", "like_count", "comment_count"]:
        d[col] = pd.to_numeric(d[col], errors="coerce").fillna(0)
    d["log_views"]    = np.log1p(d["view_count"])
    d["log_likes"]    = np.log1p(d["like_count"])
    d["log_comments"] = np.log1p(d["comment_count"])
    return d

try:
    cb_model = CatBoostClassifier()
    cb_model.load_model("catboost_ranker.cbm")
    print("\nCatBoost model loaded.")
except Exception:
    print("\nCatBoost model not found — train it in Step 7 first.")
    cb_model = None

if cb_model is not None:
    # Берём кандидатов (не из train)
    candidate_df = videos_df.iloc[indices_arr[0]].copy()
    candidate_df = candidate_df[~candidate_df["video_id"].isin(train_ids)]

    cand_positions  = candidate_df.index.tolist()
    cand_embeddings = all_embeddings[cand_positions]
    cand_sims = cosine_similarity(cand_embeddings, user_embedding).flatten()

    cand_feat = make_features(candidate_df, cand_sims)
    cand_feat["rank_score"] = cb_model.predict_proba(cand_feat[FEATURES])[:, 1]
    cand_feat["video_id"]   = candidate_df["video_id"].values

    catboost_recs = (
        cand_feat.sort_values("rank_score", ascending=False)["video_id"]
        .tolist()
    )

    print("\n══════════════════════════════════════")
    print("  CATBOOST RANKING")
    print("══════════════════════════════════════")
    evaluate(catboost_recs, test_ids)


if cb_model is not None:
    print("\n── Feature Importance ───────────────")
    fi = pd.Series(
        cb_model.get_feature_importance(),
        index=FEATURES,
    ).sort_values(ascending=False)
    for feat, val in fi.items():
        bar = "█" * int(val / 2)
        print(f"  {feat:<22} {val:5.1f}%  {bar}")


Matched watched videos: 408
Train watched: 326  |  Test watched: 82

══════════════════════════════════════
  BASELINE (cosine similarity)
══════════════════════════════════════

   K   Hit Rate   Precision    Recall      NDCG
----------------------------------------------
  10      0.000      0.0000    0.0000    0.0000
  20      0.000      0.0000    0.0000    0.0000
  50      0.000      0.0000    0.0000    0.0000

CatBoost model loaded.

══════════════════════════════════════
  CATBOOST RANKING
══════════════════════════════════════

   K   Hit Rate   Precision    Recall      NDCG
----------------------------------------------
  10      1.000      0.2000    0.0244    0.2025
  20      1.000      0.1000    0.0244    0.1307
  50      1.000      0.0600    0.0366    0.0867

── Feature Importance ───────────────
  log_views               26.4%  █████████████
  log_comments            22.8%  ███████████
  duration_seconds        19.6%  █████████
  log_likes               16.4%  ████████
  si